# Day 15 — Relational Thinking and First Queries: Tables, Keys, Data Types, psql, CREATE TABLE, INSERT, SELECT, WHERE, ORDER BY & LIMIT (8 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: think about data the way a relational database does (tables, rows, columns, keys), and use PostgreSQL to create a table, fill it and ask it precise questions with SELECT, WHERE, ORDER BY and LIMIT.

- Deliverable: day15/schema.sql (the content model of a headless CMS as two tables) and day15/queries.sql (ten customer questions answered in SQL), loaded with the given day15/seed.sql. Plus day15/dsa.py.

- Where this sits: Week 3, Phase 1 (Foundation), day 1 of 7. Tomorrow adds counting and grouping on the same table, Day 17 joins tables, Day 18 loads your CMS export from Python, and Week 4 puts a FastAPI app on top. Week 6's RAG app stores its vectors in Postgres too.

Time split (5h): the Week 3 split from the Day 14 plan, with the catch-up hour kept.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, typing and running every example in psql |
| SQL hands-on | 2h | schema.sql, then the ten questions in queries.sql |
| Catch-up | 1h | Day 5: support_ticket.py until the SLA breach report prints |
| DSA | 0.5h | paginate and duplicate_slugs in dsa.py |

Open items first: GitHub main still has one commit, from 28 Oct. Day 4's files are the starters (8 functions still raise NotImplementedError) and the Day 5 to 14 folders are not in the repo. Today's folder does not depend on them, so start here, and use the catch-up hour for Day 5. Push at the end of the day even if something is unfinished, with a message like "Day 15: WIP, schema done, Q7 to Q10 pending".

Before you start: PostgreSQL must be running. On your Mac, open Postgres.app (or run brew services start postgresql@18 if you installed it with Homebrew), then run createdb fde once in the terminal. Every example below uses that database. The examples were run on PostgreSQL 16; everything in this session behaves the same on 18.

## Recap (2 minutes, from memory)

Three questions from the Day 14 review. Answer out loud before you look anything up; the answers are on the last page.

- You call list(gen) twice on the same generator. What does the second call return, and why does that matter when you stream a large CMS export?

- In grep ERROR app.log | sort | uniq -c, why does sort have to come before uniq -c?

- A pull request merged yesterday added a .env file with a customer's API key. What is the first thing you do, and why that first?

## Concept 1: Relational thinking: tables, rows, columns and keys

### Why it matters for an FDE

Almost every customer system you will plug AI into keeps its records in a relational database: the CRM, the order system, the ticketing tool, the analytics warehouse. In the first week of a deployment you will be handed a connection string and a schema you have never seen, and you have to read it quickly and ask it correct questions. Relational thinking is the mental model that makes that possible.

### How it works

A table is a named set of rows that all have the same columns. A column has a name and exactly one data type, and every row has a value for every column (or the special marker NULL, which means "unknown"). A row is one record. You already know this shape from Contentstack: a content type defines the fields, and each entry is one filled-in set of those fields.

The big difference from JSON is that a table is strict. In a list of dicts, one entry can have a price key and the next can forget it, or store it as a string. In a table, the column price exists for every row and holds only numbers. The database refuses bad data at the door instead of letting it break your code later.

| Headless CMS (Contentstack) | Python (Days 3 and 4) | PostgreSQL |
|---|---|---|
| Content type, e.g. Blog Post | the shape you expect each dict to have | table (its definition is the schema) |
| Field, e.g. title | a dict key | column, with a fixed data type |
| Entry | one dict in a list | row |
| Entry uid, e.g. blt1a01 | a value you look things up by | primary key |
| Reference field (entry points to an author) | an id stored in another dict | foreign key (Day 17) |
| Query API: ?query={"status":"published"} | a loop with an if | SELECT ... WHERE status = 'published' |

A primary key is the column (or columns) whose value identifies exactly one row, never repeats and never changes. It is how every other system points at that row. Two kinds are common. A natural key comes from the real world, like a content type uid blog_post. A surrogate key is a number the database makes up, like id 1, 2, 3, used because no real-world value is guaranteed unique and stable. Titles and emails are bad primary keys: people edit them.

"Relational" means the data is split into several focused tables that point at each other through keys, instead of one big nested document. Today you work with single tables; on Day 17 you link them. The other big idea: SQL is declarative. You describe the result you want, and the database works out how to get it. Compare the two halves of this example.

In [ ]:
# Concept 1: the same question, asked imperatively (Python) and declaratively.
entries = [
    {"uid": "blt1", "title": "Pricing", "status": "published", "views": 900},
    {"uid": "blt2", "title": "Careers", "status": "draft", "views": 0},
    {"uid": "blt3", "title": "Home", "status": "published", "views": 4000},
]

# Imperative: YOU write HOW, step by step (filter, sort, slice)
published = [e for e in entries if e["status"] == "published"]
published.sort(key=lambda e: e["views"], reverse=True)  # most viewed first
print([e["title"] for e in published[:1]])  # -> ['Home']

# Declarative: you say WHAT you want, the database decides how.
# This string is the SQL you will write today (it runs in psql, not here):
query = """
SELECT title FROM entries
WHERE status = 'published'
ORDER BY views DESC
LIMIT 1;
"""
# REMEMBER: a table is a list of same-shaped records; SQL describes the
# result, the database plans the loop.

### Real-world FDE example

Acme Retail runs its website on Contentstack and wants an internal assistant that answers "which product pages are stale?". The CMS API returns nested JSON, page by page, and is rate limited, so the team syncs entries into a Postgres table every night. Your first job on the project is to design that table: one row per entry per locale, id as a surrogate key because the same uid appears once per locale, and columns with types the dashboard and the assistant can trust.

### Common mistakes

- Storing a list in one column, such as tags as the text 'sale,diwali'. You can no longer filter or count tags reliably. Lists become their own table (Day 17).

- Using a value people can edit (a title, an email, a slug) as the primary key. When it changes, everything that pointed at it breaks.

- Thinking of rows as having an order. A table is a set: rows come back in no guaranteed order unless you ask for one (Concept 6).

## Concept 2: PostgreSQL and psql

### Why it matters for an FDE

PostgreSQL (Postgres) is the default database at most AI startups and a common one in enterprises, and with the pgvector extension it is also where many RAG systems keep their embeddings. psql is its command-line client. On a customer's locked-down server there is often no GUI, only a terminal, and an FDE who can open psql, look around and answer a question in two minutes earns trust in the first meeting.

### How it works

Postgres is a server: a program that runs in the background, owns the data files and listens on port 5432 (Day 9's ports). Clients connect to it and send SQL as text, the same way a browser talks to a Node server. psql is one client; on Day 18 your Python code becomes another. One server holds several databases (today: fde), and each database holds tables in a schema called public by default.

```sql
# Concept 2: make a database and open psql on it (Mac terminal).
createdb fde  # one-time: an empty database called fde
psql -d fde  # connect; the prompt shows the database name
# -> psql (18.x)
# -> Type "help" for help.
# -> fde=#
# FDE TIP: on a customer system you connect with a full connection string,
# and never type the password into the command (it lands in shell history):
# psql "host=db.acme.internal port=5432 dbname=cms user=readonly
#       sslmode=require"
# psql then asks for the password, or reads it from ~/.pgpass.
```

Inside psql you type two kinds of thing. SQL statements go to the server and must end with a semicolon. Meta-commands start with a backslash, are handled by psql itself, and take no semicolon. If you press Enter without a semicolon, psql waits for the rest of the statement and the prompt changes from fde=# to fde-#.

```sql
-- Concept 2: the psql commands you will use every day (type at fde=#).
\conninfo   -- which database, user and port am I connected to?
\l          -- list databases on this server
\dt         -- list tables in the current database
\d posts    -- describe one table: columns, types, NULL rules, defaults
\x          -- toggle expanded output: one column per line, for wide rows
\timing     -- print how long each query takes (useful on Day 19)
\i day15/schema.sql  -- run a file of SQL, like pasting it in
\?          -- help for meta-commands;  \h SELECT  shows SQL syntax help
\q          -- quit (Ctrl+D works too)

SELECT 1 + 1;  -- SQL needs the semicolon; meta-commands do not
-- ->  ?column?
-- -> ----------
-- ->         2
-- -> (1 row)
-- GOTCHA: forgot the ;? The prompt shows fde-# and waits. Type ; and Enter.
```

You will mostly run SQL from files, so your work is saved and can be committed. psql -d fde -f schema.sql runs a file from the terminal, and adding -v ON_ERROR_STOP=1 makes psql stop at the first error and exit with a non-zero code, the same idea as Day 9's exit codes. Without it psql reports the error and carries on with the next statement, which is how one typo turns into twenty confusing errors.

### Real-world FDE example

A bank gives you read-only access to a reporting replica for a two-week pilot. Before writing any code you run \dt to see the tables, \d on the three that sound relevant, and SELECT * FROM tickets LIMIT 5; to see real values. Twenty minutes in psql tells you that dates are stored as text and half the priority values are blank. That changes your plan before you have built anything on top of it.

### Common mistakes

- Typing a meta-command with a semicolon or SQL without one. Backslash means psql; everything else needs ;.

- Connecting to the wrong database. Run \conninfo before you change anything, and get read-only credentials for production.

- Running a long SQL file without ON_ERROR_STOP=1 and reading only the last line of output. The first error is the one that matters.

## Concept 3: Data types and CREATE TABLE

### Why it matters for an FDE

JavaScript has one number type and treats almost anything as a string. A database makes you choose, and the choice decides whether totals add up, dates sort correctly and an invoice is right to the paisa. Many data bugs you will be called in to fix come from a wrong type chosen years ago.

### How it works

These types cover nearly everything in the first weeks:

| Type | Holds | Use it for | JS or CMS equivalent |
|---|---|---|---|
| integer / bigint | whole numbers (bigint for very large ones) | counts, views, surrogate ids | number |
| numeric(10, 2) | exact decimals: 10 digits, 2 after the point | money, anything that must add up exactly | a price field (stored exactly) |
| text | a string of any length | titles, slugs, uids, locales | string; a single-line or multi-line text field |
| boolean | true, false (or NULL) | flags like is_featured | boolean field |
| date | a calendar day | birthdays, due dates | a date field without time |
| timestamptz | an exact moment in time | created_at, published_at, updated_at | an ISO date string with an offset |

CREATE TABLE defines a table: its name, then each column with a type and optional rules. PRIMARY KEY marks the key. NOT NULL makes a value required. DEFAULT gives the value to use when an INSERT leaves the column out. GENERATED ALWAYS AS IDENTITY makes the database hand out 1, 2, 3 for a surrogate key. Run this in psql with \i, or paste it in:

```sql
-- Concept 3: a table for blog posts in a CMS.
DROP TABLE IF EXISTS posts;  -- start clean so the file can be re-run

CREATE TABLE posts (
    -- the database numbers each row for you: 1, 2, 3 ...
    id           integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    uid          text NOT NULL,          -- like a Contentstack entry uid
    title        text NOT NULL,          -- NOT NULL: a value is required
    locale       text NOT NULL DEFAULT 'en-us',  -- used when omitted
    status       text NOT NULL DEFAULT 'draft',
    word_count   integer,                -- no NOT NULL: may be unknown
    is_featured  boolean NOT NULL DEFAULT false,
    published_at timestamptz,            -- NULL until it goes live
    created_at   timestamptz NOT NULL DEFAULT now()  -- time of INSERT
);  -- no comma after the last column
-- -> NOTICE:  table "posts" does not exist, skipping   (first run only)
-- -> DROP TABLE
-- -> CREATE TABLE
```

Check what you built with \d posts. Read it column by column: the type, whether NULL is allowed, and the default.

```sql
-- Concept 3: describe the table (Collation column dropped to fit).
\d posts
-- ->    Column     |           Type           | Nullable |    Default
-- -> --------------+--------------------------+----------+--------------
-- ->  id           | integer                  | not null | generated
-- ->               |                          |          | always as
-- ->               |                          |          | identity
-- ->  uid          | text                     | not null |
-- ->  title        | text                     | not null |
-- ->  locale       | text                     | not null | 'en-us'::text
-- ->  status       | text                     | not null | 'draft'::text
-- ->  word_count   | integer                  |          |
-- ->  is_featured  | boolean                  | not null | false
-- ->  published_at | timestamp with time zone |          |
-- ->  created_at   | timestamp with time zone | not null | now()
-- -> Indexes:
-- ->     "posts_pkey" PRIMARY KEY, btree (id)
-- The primary key got an index for free: lookups by id are fast (Day 19).
```

A word on timestamptz. It stores one exact moment, and psql shows it in your session's time zone. SET timezone = 'Asia/Kolkata'; shows IST; a server in Virginia might show UTC. The moment is the same, only the display changes, which is why the starter files begin with that SET line. Plain timestamp (without tz) stores a wall-clock time with no zone, and it causes bugs as soon as two offices or a cloud server are involved.

### Real-world FDE example

A retailer's old export stored prices as real (a float). Their nightly report showed a Diwali bundle at 1949.9999 and finance stopped trusting the dashboard. The fix was a migration to numeric(10, 2). On a new project, choose numeric for money and timestamptz for times on day one, and that whole class of incident never happens.

### Common mistakes

- Money in real or double precision. Floats are approximate (Day 1: 0.1 + 0.2). Use numeric.

- Dates stored as text like '08/11/2026'. They sort wrongly and mean different days in India and the US. Use date or timestamptz.

- Writing column names in double quotes, like "Title". Postgres then treats the name as case-sensitive forever, and every query must quote it. Use lowercase snake_case and no quotes.

- A comma after the last column, or a missing comma between two. Read the error's LINE and caret: it points at the spot.

- Running DROP TABLE against a real database. It deletes the table and all its rows with no undo. In your own practice files it is fine; on a customer system, never.

## Concept 4: INSERT, quoting and RETURNING

### Why it matters for an FDE

Every integration ends with writing rows somewhere: synced entries, ticket summaries an LLM produced, evaluation results. You need to know exactly what the database does with each value you send, which values it fills in for you, and how to get those values back.

### How it works

INSERT INTO table (columns) VALUES (values); adds rows. List the columns you are filling, in any order, and give values in the same order. Columns you leave out get their DEFAULT, or NULL if they have none. Strings and timestamps go in single quotes; numbers, true, false and NULL go bare. A single quote inside a string is written twice.

```sql
-- Concept 4: add rows. Strings use SINGLE quotes in SQL.
INSERT INTO posts (uid, title, word_count)
VALUES ('blt001', 'Welcome to our new site', 420);
-- -> INSERT 0 1   (the 1 is the number of rows; ignore the 0)

-- several rows in one statement: one round trip, all or nothing
INSERT INTO posts (uid, title, locale, status, word_count, published_at)
VALUES
    ('blt002', 'Pricing explained', 'en-us', 'published', 980,
     '2026-10-01 09:30:00+05:30'),  -- a timestamp is written as text
    ('blt003', 'Precios explicados', 'es-es', 'published', 1010,
     '2026-10-03 10:00:00+05:30'),
    -- '' inside a string is one quote; NULL is bare, never 'NULL'
    ('blt004', 'Ravi''s launch notes', 'en-us', 'draft', NULL, NULL),
    ('blt005', 'Careers at Acme', 'en-gb', 'published', 300,
     '2026-09-15 12:00:00+05:30');
-- -> INSERT 0 4

-- RETURNING hands back what the database filled in (id, defaults)
INSERT INTO posts (uid, title, is_featured)
VALUES ('blt006', 'Holiday sale', true)
RETURNING id, uid, locale, status;
-- ->  id |  uid   | locale | status
-- -> ----+--------+--------+--------
-- ->   6 | blt006 | en-us  | draft
-- -> (1 row)
-- REMEMBER: leave a column out and you get its DEFAULT (or NULL);
-- RETURNING shows you what that was.
```

Now three inserts that fail. Each error message tells you exactly what was wrong, so read it before you change anything.

```sql
-- Concept 4: three inserts the database refuses, and why.
INSERT INTO posts (id, uid, title) VALUES (99, 'x', 'y');
-- -> ERROR:  cannot insert a non-DEFAULT value into column "id"
-- -> DETAIL:  Column "id" is an identity column defined as GENERATED ALWAYS.
-- The database owns the id. Leave it out and let it count.

INSERT INTO posts (uid) VALUES ('blt007');
-- -> ERROR:  null value in column "title" of relation "posts" violates
-- ->         not-null constraint
-- NOT NULL did its job: no title, no row.

INSERT INTO posts (uid, title) VALUES ("blt008", "Oops");
-- -> ERROR:  column "blt008" does not exist
-- GOTCHA: double quotes mean a NAME (column or table) in SQL, not text.
-- In JS 'a' and "a" are the same string; in SQL only 'a' is a string.
```

One surprise: the failed blt007 insert still used up id 7, so the next successful row gets id 8. Identity numbers are unique, not gapless. Never use them to count rows or to mean "the 7th post".

### Real-world FDE example

Your sync job inserts a ticket summary that Claude generated, then needs the new row's id to store the summary's citations in a second table. INSERT ... RETURNING id gives you the id in the same round trip, with no second query and no guessing "the highest id is probably mine", which breaks as soon as two jobs run at once.

### Common mistakes

- Double quotes for text values. Use single quotes for values, and keep double quotes for names (which you will rarely need).

- Writing 'NULL' in quotes. That is the four-letter string NULL, not a missing value.

- Leaving out the column list: INSERT INTO posts VALUES (...). It depends on the column order and breaks when someone adds a column. Always name the columns.

- Building an INSERT by pasting user input into a string. That is SQL injection; Day 18 shows the safe way from Python.

## Concept 5: SELECT and WHERE

### Why it matters for an FDE

SELECT is the statement you will write most often in your career. In a discovery call, "how many of your product pages are missing a price?" answered live in a minute is worth more than a slide. Later, when an LLM writes SQL for a customer (a common agent tool), you are the one who checks that its WHERE clause really means what the user asked.

### How it works

SELECT columns FROM table WHERE condition; returns the rows for which the condition is true, with only the columns you listed. * means all columns, which is fine for a first look and bad in code you keep. You can compute new columns and name them with AS.

```sql
-- Concept 5: choose columns, compute one, filter rows.
SELECT uid, title, status FROM posts;  -- 6 rows, 3 columns
-- (output not shown: all six posts)

-- a computed column, named with AS (like a derived prop in React)
SELECT uid, word_count, word_count / 200 AS read_minutes FROM posts;
-- ->   uid   | word_count | read_minutes
-- ->  blt001 |        420 |            2   <- integer / integer drops the
-- ->  blt002 |        980 |            4      decimals, as // does in Python
-- ->  blt003 |       1010 |            5
-- ->  blt004 |            |                 <- NULL in, NULL out
-- ->  blt005 |        300 |            1
-- ->  blt006 |            |

-- WHERE keeps only the rows where the condition is TRUE
SELECT uid, title FROM posts WHERE status = 'published';
-- ->  blt002 | Pricing explained
-- ->  blt003 | Precios explicados
-- ->  blt005 | Careers at Acme
SELECT uid FROM posts WHERE status = 'Published';
-- -> (0 rows)   GOTCHA: = on text is case-sensitive
```

Conditions use = (one equals sign, not JavaScript's ===), <> or != for "not equal", and <, <=, >, >=. Combine them with AND, OR and NOT. A few shortcuts make conditions shorter and easier to read:

```sql
-- Concept 5: combining conditions, and the shortcuts.
SELECT uid, locale FROM posts
WHERE status = 'published' AND locale <> 'en-us';
-- ->  blt003 | es-es
-- ->  blt005 | en-gb

-- BETWEEN includes both ends: same as >= 400 AND <= 1000
SELECT uid, word_count FROM posts WHERE word_count BETWEEN 400 AND 1000;
-- ->  blt001 |        420
-- ->  blt002 |        980

-- IN: like ['en-gb', 'es-es'].includes(locale) in JS
SELECT uid, locale FROM posts WHERE locale IN ('en-gb', 'es-es');
-- ->  blt003 | es-es
-- ->  blt005 | en-gb

-- LIKE matches a pattern: % = any characters, _ = exactly one.
-- ILIKE is the case-insensitive version (Postgres only).
SELECT uid, title FROM posts WHERE title ILIKE '%pric%';
-- ->  blt002 | Pricing explained   ("Precios" has no "pric", so no match)
```

Now the most important trap of the day. NULL means "unknown", so any comparison with it is also unknown, never true. WHERE word_count = NULL asks "is it equal to an unknown value?", the answer is unknown, and the row is dropped. To test for a missing value you must write IS NULL or IS NOT NULL. Day 16 goes deeper into NULL.

```sql
-- Concept 5: NULL, and AND/OR precedence.
SELECT uid FROM posts WHERE word_count = NULL;
-- -> (0 rows)   never true, not even for rows with no word count
SELECT uid FROM posts WHERE word_count IS NULL;
-- ->  blt004
-- ->  blt006
-- REMEMBER: test for missing values with IS NULL, never = NULL.

-- AND binds tighter than OR, like * before + in maths.
-- This reads: draft OR (featured AND Spanish)
SELECT uid FROM posts
WHERE status = 'draft' OR is_featured AND locale = 'es-es';
-- ->  blt001, blt004, blt006   (all three drafts)
-- Brackets say what you mean: (draft OR featured) AND Spanish
SELECT uid FROM posts
WHERE (status = 'draft' OR is_featured) AND locale = 'es-es';
-- -> (0 rows)
```

It helps to know the order the database works in, which is not the order you write the clauses. It takes the rows FROM the table, keeps those that pass WHERE, then builds the SELECT columns, then sorts (ORDER BY, Concept 6), then cuts (LIMIT). That order explains several errors you will meet today.

### Real-world FDE example

In a scoping call, Acme's content lead says "most of our product pages have prices". You run SELECT uid, title FROM entries WHERE content_type = 'product' AND price IS NULL; while she is still talking, and find nine. That number goes into your proposal as a data-cleanup step before the AI assistant can answer price questions, and it came from evidence, not a guess.

### Common mistakes

- = NULL instead of IS NULL. It returns nothing and raises no error, so the bug hides.

- Mixing AND and OR without brackets. Add brackets every time you combine them, even when you think you know the precedence.

- Case mismatches like 'Published' versus 'published'. Use ILIKE, or store values in one agreed case (a rule you will enforce on Day 19).

- SELECT * in code you keep. Name the columns, so a new column cannot break your output or leak data you did not mean to show.

## Concept 6: ORDER BY, LIMIT and OFFSET

### Why it matters for an FDE

"The latest 10 blog posts", "the 5 most viewed pages", "page 3 of the search results": nearly every API endpoint and dashboard tile you build in Week 4 is a query with an order and a limit. Getting the order wrong is a quiet bug: the page loads, but shows the wrong items.

### How it works

A table has no order of its own. Without ORDER BY, Postgres returns rows in whatever order is cheapest, and that order can change after an update or a restart. ORDER BY column sorts ascending; add DESC for descending. List several columns to break ties: the second is used only when the first is equal. LIMIT n keeps the first n rows after sorting, and OFFSET n skips n rows first, which is how pages work.

```sql
-- Concept 6: sorting. Most words first.
SELECT uid, word_count FROM posts ORDER BY word_count DESC;
-- ->  blt004 |              <- NULLs come FIRST with DESC in Postgres
-- ->  blt006 |                 (these two tie: their order may vary)
-- ->  blt003 |       1010
-- ->  blt002 |        980
-- ->  blt001 |        420
-- ->  blt005 |        300

SELECT uid, word_count FROM posts ORDER BY word_count DESC NULLS LAST;
-- ->  blt003 | 1010, blt002 | 980, blt001 | 420, blt005 | 300,
-- ->  then blt004 and blt006 at the end

-- two keys: status A to Z, then, within each status, most words first
SELECT uid, status, word_count FROM posts
ORDER BY status, word_count DESC NULLS LAST;
-- ->  blt001 | draft     |  420
-- ->  blt004 | draft     |
-- ->  blt006 | draft     |
-- ->  blt003 | published | 1010
-- ->  blt002 | published |  980
-- ->  blt005 | published |  300
```

Paging is the same idea: sort first, then skip and keep. Page p of size n is LIMIT n OFFSET (p - 1) * n.

```sql
-- Concept 6: LIMIT and OFFSET, i.e. pages (2 rows per page).
SELECT uid, title FROM posts ORDER BY id LIMIT 2;           -- page 1
-- ->  blt001 | Welcome to our new site
-- ->  blt002 | Pricing explained
SELECT uid, title FROM posts ORDER BY id LIMIT 2 OFFSET 2;  -- page 2
-- ->  blt003 | Precios explicados
-- ->  blt004 | Ravi's launch notes
-- OFFSET = (page - 1) * page_size. Same idea as Contentstack's
-- ?limit=2&skip=2 on the Content Delivery API.

-- "the latest published post": sort newest first, keep one
SET timezone = 'Asia/Kolkata';  -- show times in IST
SELECT uid, title, published_at FROM posts
WHERE status = 'published'
ORDER BY published_at DESC
LIMIT 1;
-- ->  blt003 | Precios explicados | 2026-10-03 10:00:00+05:30
-- REMEMBER: no ORDER BY, no order. LIMIT without ORDER BY is a random
-- sample, not "the first rows".
```

Remember the order of work from Concept 5: FROM, WHERE, SELECT, ORDER BY, LIMIT. Because ORDER BY runs after SELECT, it can use a column alias. WHERE runs before SELECT, so it cannot:

```sql
-- Concept 6: an alias works in ORDER BY but not in WHERE.
SELECT uid, title AS headline FROM posts ORDER BY headline LIMIT 3;
-- ->  blt005 | Careers at Acme
-- ->  blt006 | Holiday sale
-- ->  blt003 | Precios explicados
SELECT uid, title AS headline FROM posts WHERE headline LIKE 'P%';
-- -> ERROR:  column "headline" does not exist
-- WHERE ran before SELECT made the alias. Write WHERE title LIKE 'P%'.
```

### Real-world FDE example

A customer's "latest news" widget showed the same article on page 1 and page 2. The query sorted by published_at only, and three articles shared the same timestamp from a bulk publish, so Postgres was free to return them in a different order on each request. Adding a tie-breaker, ORDER BY published_at DESC, id DESC, fixed it. Every paginated query needs an order that is unique.

### Common mistakes

- LIMIT without ORDER BY and calling it "the first 10". It is any 10.

- Sorting by a column that can tie and paginating on it. Add a unique column, such as id, as the last sort key.

- Forgetting that NULLs sort first with DESC. Add NULLS LAST when the missing values should go to the end.

- Using a big OFFSET for deep pages. The database still reads and throws away every skipped row, so page 5,000 is slow. Fine for today; a faster pattern comes with indexes in Week 4.

## Check your understanding (predict first, then run)

Write your prediction for each query in the notebook, then run it in psql against the posts table from Concepts 3 and 4. If your posts table has changed since then, re-run those two examples first. If a result surprises you, find the word in the query that caused it.

```sql
-- Check your understanding: predict each result, then run it.
-- 1. Which rows, and why not the others?
SELECT uid FROM posts WHERE status = 'draft' AND word_count > 400;

-- 2. Which two uids, in which order?
SELECT uid FROM posts WHERE NOT is_featured ORDER BY id DESC LIMIT 2;

-- 3. What are a and b? (Look at Concept 5's read_minutes.)
SELECT 'It''s' AS txt, 7 / 2 AS a, 7 / 2.0 AS b;

-- 4. Which rows, and in which order?
SELECT uid, title FROM posts WHERE title ILIKE 'p%' ORDER BY title;

-- 5. Which id does the last INSERT return?
DROP TABLE IF EXISTS t;
CREATE TABLE t (
    id   integer GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name text NOT NULL
);
INSERT INTO t (name) VALUES ('a');
INSERT INTO t (name) VALUES (NULL);
INSERT INTO t (name) VALUES ('b') RETURNING id;

-- 6. How many rows? Rewrite it with brackets so it means what it says.
SELECT uid FROM posts
WHERE locale = 'en-us' OR locale = 'en-gb' AND status = 'published'
ORDER BY id;
```

## Hands-on task (about 2 hours): day15/schema.sql and day15/queries.sql

The scenario: Acme Retail syncs its Contentstack stack into Postgres so a reporting dashboard (and, in Week 6, an AI assistant) can query it. You design the two tables and answer the content team's first ten questions. The starter files are in your repo under day15/: the brief and acceptance criteria are at the top of each file, with hints above every TODO. seed.sql is complete; do not edit it, because the expected answers depend on it.

- Pull and look (10 min). git pull, then read the top of schema.sql and queries.sql in VS Code. Open seed.sql and look at the data: three content types with prices only on products, Hindi and Marathi entries that share a slug with their English versions, drafts with no author.

- Write schema.sql (40 min). Drop both tables if they exist (entries first), create content_types and entries exactly as the brief says, then insert the four content types in one INSERT ... RETURNING uid.

- Load it (5 min). From inside day15/, run psql -d fde -v ON_ERROR_STOP=1 -f schema.sql -f seed.sql. If seed.sql fails, the error names the column that does not match the brief: fix schema.sql and run the command again. Then run it a second time to prove the file is safe to re-run.

- Check the shape (5 min). In psql: \d content_types, \d entries, and SELECT count(*) FROM entries; (expect 20). count(*) is tomorrow's topic; today just read the number.

- Answer Q1 to Q10 (60 min). Write one query per question in queries.sql. Run each one alone in psql first, compare it with its Expected comment, then save it in the file. When all ten are in, run psql -d fde -f queries.sql and check every result once more.

- Commit and push. git add day15 && git commit -m "Day 15: CMS schema and 10 queries" && git push.

Acceptance criteria:

- psql -d fde -v ON_ERROR_STOP=1 -f schema.sql -f seed.sql runs with no ERROR lines, twice in a row.

- \d entries shows the 13 columns with the types, NULL rules and defaults in the brief; SELECT count(*) FROM entries; returns 20.

- Every query in queries.sql matches its Expected comment: row count, first row and (where given) last row.

- No SELECT *; every multi-row query has an ORDER BY with a tie-breaker where values can repeat.

- Each statement has a one-line comment saying why, written for a customer.

- The day15 folder is pushed to main.

Hints, if you are stuck (try for 10 minutes first):

- Q2 and Q3: ORDER BY a DESC, b sorts by a, and uses b only when two rows have the same a.

- Q4: drafts with no author is two conditions. One of them tests for a missing value; Concept 5 says how.

- Q8: "under 1000" means <, not <=, and the question does not mention status.

- Q9: page 2 at 5 per page skips how many rows?

- Q10: compare updated_at with the date '2026-10-01'. The SET timezone line at the top of the file decides which midnight that is.

Catch-up lane (1h): Day 5, support_ticket.py. Open the Day 5 folder from the files in the project chat (or your laptop copy) and finish the Ticket dataclass and the SLA report. Done when the SLA breach report prints for tickets_export.json as shown in the file's docstring. Push it as its own commit: "Day 5: support ticket model".

## DSA practice (30 min)

Two problems in day15/dsa.py that are today's SQL written by hand. Signatures, examples and hints are in the file; the expected outputs are in its __main__ block.

- paginate(items, page, page_size). Return one page of a list, like LIMIT and OFFSET. Hint: work out how many items come before page 1, page 2 and page 3, and turn that into a formula. Slicing past the end of a list does not raise an error.

- duplicate_slugs(rows). Find (slug, locale) pairs used by more than one entry: two pages at the same URL. Hint: group uids in a dict whose key is a tuple, as on Day 3, then keep only the keys with more than one uid. The same slug in two different locales is not a duplicate.

## Quiz (3 quick questions)

- A customer's CMS has a tags field holding a list on each entry. Their developer suggests storing it in a text column as 'sale,diwali'. What do you say, and why?

- Write the query: the 3 most recently published en-us entries in the entries table, showing title and published_at.

- Why does WHERE author = NULL return no rows even when some authors are missing, and what do you write instead?

## What you achieved today

You learned the relational model (tables, typed columns, rows and primary keys), set up Postgres and psql, chose data types for a real content model, created tables, inserted rows and read the errors the database gives back, and asked precise questions with SELECT, WHERE, ORDER BY and LIMIT. The CMS model in your repo is the table that Days 16 to 20 build on.

Tomorrow (Day 16): aggregates and NULL: COUNT, SUM, AVG, GROUP BY, HAVING, DISTINCT, COALESCE and CASE, turned into a content-ops report on today's entries table.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 15 review": your recap answers, your six predictions next to the real outputs, your schema.sql and queries.sql (or the commit link), your dsa.py, your quiz answers and the Day 5 status. Note anything that felt unclear; tomorrow starts there.

## Recap answers

*Your answers:*
